# graphArt_LM: the full-rules two-step model, a full epoch on two GPUs

The last full-rules run (9 of 21 torchvision graphs right) hit the time cap after about
65% of an epoch. This is the same run -- the same data, prompts, and settings, a fresh
LoRA on `Qwen2.5-Coder-1.5B-Instruct` -- trained on both T4s at once, so the epoch
finishes. Then it scores the 40 test programs and the 21 torchvision samples, step 2
given the model's own step-1 nodes. About 2 hours, capped at 2.5.

**Before running**: *GPU T4 x2* (both are used), *Internet on*. No inputs needed. Then
*Save Version -> Save & Run All (Commit)*.

**When it finishes**, download `graphart_finetune_two_step_epoch.zip` from the *Output*
tab: the results, the adapter, and the scoring logs.

Generated by `scripts/make_kaggle_notebook.py --finetune-two-gpu`. Edit that script, not this file.

In [ ]:
# BASE_MODEL   tuned from scratch (a fresh LoRA), like every earlier session.
# ADAPTER      its name; results land in results/hf__<ADAPTER>__two_step*.jsonl.
# GPUS, EFFECTIVE_BATCH  training is split across both T4s; each step is still 16
#              examples (8 per GPU), as in every earlier run.
# TRAIN_ARGS   --two-step --full-prompts: two examples per program (nodes, then edges),
#              full-rules prompts. train/8 is every eighth tier 1-4 program (40 a tier);
#              all of tiers 5-6; val/8 and half of large_val. --max-len 6144 leaves out
#              the 10 longest tier-6 examples (of 640), whole -- 8192 did not fit last time.
# FALLBACK_MAX_LEN  if even 6144 does not fit in GPU memory, train with this instead.
# MEMORY_CHECK forward + backward on this many of the longest examples, before anything else.
# EVALS        (data set, reply cap per call), each program in two calls with the same full
#              prompts: the 40 stratified test programs and the 21 torchvision samples.
#              The two jobs run at the same time, one on each GPU.
# EVAL_RESERVE_HOURS  kept back from training for EVALS (they took ~15 min last time).
# VALIDATE_ONLY True: only check the code arrived intact (works on a CPU session).
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"; ADAPTER = "qwen2.5-coder-1.5b-graphart-lora-two-step-full-epoch"; EPOCHS = 1.0; GPUS = 2; EFFECTIVE_BATCH = 16; TRAIN_ARGS = ["--two-step", "--full-prompts", "--train-split", "train/8,large_train", "--val-split", "val/8,large_val/2", "--max-len", "6144", "--drop-over-max", "--eval-steps", "40"]; FALLBACK_MAX_LEN = 5120; MEMORY_CHECK = 6; SMOKE_STEPS = 8; EVALS = [("test", 3000), ("torchvision", 4500)]; EVAL_RESERVE_HOURS = 0.5; TIME_BUDGET_HOURS = 2.5; EVAL_ARGS = []; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "5b9fffec4cd8bd26e5ea6149da339157188837b3d45ae8ee1371be0a2c4b527c"
BUNDLE = (
    "..."
    "WVo="
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:xz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install the pinned libraries

In [ ]:
from importlib.metadata import PackageNotFoundError, version

PINS = {'transformers': '5.20.2', 'peft': '0.21.2', 'accelerate': '1.15.0'}

def pinned(package, wanted):
    try:
        return version(package) == wanted
    except PackageNotFoundError:
        return False

missing = [f"{p}=={v}" for p, v in PINS.items() if not pinned(p, v)]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# peft probes two optional quantisation libraries when it builds LoRA layers, and
# raises -- rather than skipping them -- if they are installed but older than it
# supports. Kaggle ships torchao 0.10 (peft wants >= 0.16), which killed a run. We
# use neither, so remove them if present.
for pkg in ("torchao", "gptqmodel"):
    try:
        print(f"removing {pkg} {version(pkg)} (peft rejects old versions; not used here)", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg], check=True)
    except PackageNotFoundError:
        pass

subprocess.run([sys.executable, "-c", "import torch, transformers, peft; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
                "'| transformers', transformers.__version__, '| peft', peft.__version__)"])

## 3. Download the base model

In [ ]:
# Its own step with a time limit: a stalled download inside training would otherwise
# sit until the session cap. ~3 GB; a few minutes on Kaggle's network.
FETCH = ("from huggingface_hub import snapshot_download; "
         f"print('downloaded to', snapshot_download({BASE_MODEL!r}))")
for attempt in (1, 2):
    try:
        subprocess.run([sys.executable, "-c", FETCH], check=True, timeout=1800)
        break
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError) as exc:
        print(f"download attempt {attempt} failed: {exc}", flush=True)
else:
    raise SystemExit("could not download the base model -- stopping before any training")

## 4. Two GPUs? Then check the data and the GPU memory

In [ ]:
import re, zipfile

n = int(subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"],
                       capture_output=True, text=True).stdout.strip() or 0)
print(f"{n} GPU(s) visible", flush=True)
if n < GPUS:
    raise SystemExit(f"this session has {n} GPU(s), not {GPUS} -- set the accelerator to GPU T4 x2")

ONE_GPU = dict(os.environ, CUDA_VISIBLE_DEVICES="0", HF_HUB_OFFLINE="1", PYTHONUNBUFFERED="1")
ALL_GPUS = dict(ONE_GPU, CUDA_VISIBLE_DEVICES=",".join(str(i) for i in range(GPUS)))
TORCHRUN = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", str(GPUS)]
ZIP_NAME = "graphart_finetune_two_step_epoch.zip"
LOGS = REPO / "logs"
LOGS.mkdir(exist_ok=True)

def remaining():
    return TIME_BUDGET_HOURS * 3600 - (time.time() - START)

def run(cmd, env):
    # Run, echo the output live, and return (exit code, output).
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    return proc.wait(), "".join(lines)

def check(*extra):
    # One process on one GPU: the data check and the memory check.
    return subprocess.run([sys.executable, "scripts/train_lora.py", "--base", BASE_MODEL, *TRAIN_ARGS, *extra],
                          env=ONE_GPU).returncode

def train(*extra):
    # Both GPUs: each step's EFFECTIVE_BATCH examples split between them.
    return run([*TORCHRUN, "scripts/train_lora.py", "--base", BASE_MODEL, *TRAIN_ARGS,
                "--grad-accum", str(EFFECTIVE_BATCH // GPUS), *extra], ALL_GPUS)

def package():
    with zipfile.ZipFile(WORK / ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as z:
        for root in (REPO / "results", REPO / "lora" / ADAPTER, LOGS):
            for p in root.rglob("*") if root.exists() else []:
                if p.is_file() and "checkpoints" not in p.parts:
                    z.write(p, p.relative_to(REPO))

def with_max_len(args, n):
    out = list(args)
    out[out.index("--max-len") + 1] = str(n)
    return out

if check("--out", "lora/check", "--check-only") != 0:
    raise SystemExit("the data check failed -- see the traceback above")
code = check("--out", "lora/check", "--memory-check", str(MEMORY_CHECK))
if code == 5:                                 # train_lora.py: out of memory
    print(f"the longest examples do not fit -- falling back to --max-len {FALLBACK_MAX_LEN}", flush=True)
    TRAIN_ARGS = with_max_len(TRAIN_ARGS, FALLBACK_MAX_LEN)
    code = check("--out", "lora/check", "--memory-check", str(MEMORY_CHECK))
if code != 0:
    raise SystemExit(f"the memory check failed (exit {code}) -- see the log above")

## 5. Smoke run on both GPUs: is it learning?

In [ ]:
# Under torchrun a failing process's own exit code is lost, so the cause is read from the
# log: "NOT LEARNING" or a non-finite loss can be fp16 trouble, worth one retry in fp32.
code, out = train("--out", "lora/smoke", "--smoke-steps", str(SMOKE_STEPS))
if code != 0 and ("NOT LEARNING" in out or "STOPPED: loss is" in out):
    print(f"smoke run failed in fp16 -- retrying in fp32", flush=True)
    code, out = train("--out", "lora/smoke", "--smoke-steps", str(SMOKE_STEPS), "--fp32")
    if code == 0:
        TRAIN_ARGS = [*TRAIN_ARGS, "--fp32"]
if code != 0:
    raise SystemExit(f"the smoke run failed (exit {code}) -- stopping before the full run")

## 6. Train on both GPUs

In [ ]:
hours = max(remaining() / 3600 - EVAL_RESERVE_HOURS, 0.25)
print(f"training for at most {hours:.2f} h", flush=True)
code, out = train("--out", f"lora/{ADAPTER}", "--epochs", str(EPOCHS), "--time-budget-hours", f"{hours:.2f}")
(LOGS / "train.log").write_text(out)
if code != 0:
    package()
    raise SystemExit("training failed -- see the log above")
print((REPO / "lora" / ADAPTER / "train_summary.json").read_text())
package()

## 7. Score it: both jobs at once, one per GPU

In [ ]:
adapter = REPO / "lora" / ADAPTER
jobs = []
for i, (split, cap) in enumerate(EVALS):
    cmd = [sys.executable, "-m", "pipeline.run_two_step", "--backend", "hf", "--model", str(adapter),
           "--split", split, "--max-tokens", str(cap), *(["--stratify", "10"] if split == "test" else []),
           *EVAL_ARGS]
    log = open(LOGS / f"score_{split}.log", "w")
    print(f"##### {split} on GPU {i % GPUS}   ({remaining() / 3600:.1f} h left)", flush=True)
    jobs.append((split, log, subprocess.Popen(cmd, env=dict(ONE_GPU, CUDA_VISIBLE_DEVICES=str(i % GPUS)),
                                              stdout=log, stderr=subprocess.STDOUT)))
for split, log, proc in jobs:
    try:
        proc.wait(timeout=max(remaining() - 180, 60))
    except subprocess.TimeoutExpired:
        proc.kill()
        print(f"{split}: stopped by the time budget -- the finished replies are kept", flush=True)
    log.close()
    print(f"\n##### {split} (exit {proc.returncode})\n" + (LOGS / f"score_{split}.log").read_text(), flush=True)
    package()
for meta in sorted((REPO / "results").glob("*.meta.json")):
    m = json.loads(meta.read_text())
    print(f"{meta.stem:80s} n={m['n']} failed={m['failed']} joined={m.get('joined')}")
print(f"done in {(time.time() - START) / 3600:.1f} h -- download {ZIP_NAME}")